## Monte Carlo Within Metropolis for Diffusion Models

### Motivation

This notebook is a continuation of `monte_carlo_within_metropolis.ipynb`, where we defined the Monte Carlo Within Metropolis (MCWM) algorithm for sampling from a posterior when likelihood evaluations require Monte Carlo estimation.

Where this notebook differs is the application of MCWM to diffusion processes. Here, we observe a process at discrete times, but its path between observations is hidden.

The latent Gaussian model (LGM) we used in `monte_carlo_within_metropolis.ipynb` contains a latent variable, leaving us with the likelihood
$$
p(y\mid\theta)=\int p(y,z\mid\theta)\,dz.
$$
We estimated this likelihood by averaging over simulated values of the latent variable. In that example, both the exact likelihood and posterior were available, so we used them to test the algorithm against a known benchmark.

Inference for diffusion models is similar in nature: we treat the unobserved path between two observed states as latent data. Let $x_i$ and $x_{i+1}$ be observations at consecutive observation times, and let $\theta$ denote the unknown diffusion parameters.

To approximate the transition density, we divide the observation interval into $M$ substeps and introduce intermediate states $z_1,\ldots,z_{M-1}$. Under this discretized model, the transition density is
$$
p_M(x_{i+1}\mid x_i,\theta)
=
\int
p_M(x_{i+1},z_1,\ldots,z_{M-1}\mid x_i,\theta)
\,dz_1\cdots dz_{M-1}.
$$
Here, $p_M$ denotes densities under the discretized model. The observed endpoints remain fixed while we integrate over the intermediate states.

MCWM can then use Monte Carlo estimates of these transition densities to perform approximate posterior sampling. This introduces an additional consideration: besides the likelihood-estimation sample size $N$, we must choose the path resolution $M$.

### In This Notebook

In this notebook, we first cover transitioning the MCWM theory and algorithmic information
presented in ```monte_carlo_within_metropolis.ipynb``` to the application of 
discretely-observed diffusion  processes. We finish by implement the theory and
algorithm to an example using the Ornstein–Uhlenbeck process.

### Theory

#### Diffusion Processes

A diffusion process is a solution to the stochastic differential equation (SDE):
$$
dX_t = \mu(X_t, \theta)dt + \sigma(X_t, \theta)\,dW_t, \quad 0 \leq t \leq T,
$$
where $X_t \in \mathbb{R}^d$, $\mu \in \mathbb{R}^{d}$ and $\nu = \sigma \sigma^{\top} 
\in \mathbb{R}^{d \times d}$ are drift and covariance coefficients respectively,
$\theta$ is the parameter vector, and $W_t$ is a $d$-dimensional Brownian motion (Wiener process).

##### Example: Ornstein-Uhlenbeck
The Ornstein-Uhlenbeck (UH) process is a mean-reverting stochastic process. Such a 
process tends to drift towards its mean function. In other words, one can see the UH
process as a random walk in continuous time but with a tendency for the walk to move back 
towards a central location, with a greater attraction when the process is further away
from the centre. 

UH is defined by the following SDE: 
$$
    d X_t = \theta(\mu - X_t)dt + \sigma\,dW_t,
$$
where $\theta$ is the mean reversion speed/strength.

#### Diffusions in Bayesian Inference and the Need for MCWM

In statistics, one may use a diffusion process as a model to describe, say, temporal-based data.
In a Bayesian setting, it becomes of interest to perform inference on the parameters $\theta$ that
describes $\mu$ and $\sigma$. The primary mechanism to do so is via the posterior density $p(\theta \mid y)$.
Unlocking the posterior allows one to quantify uncertainty about 
the estimated parameters after observing evidence. The posterior is proportional the 
likelihood of the data given the parameters multiplied by the prior of the parameters:
$$
    p(\theta \mid y) \propto p(y \mid \theta)p(\theta).
$$

When attempting to estimate the parameters $\theta$ in a continuous-time Markov process,
the reality of the world only allows us to capture observations at discrete time points
(usually taken at equally spaced time intervals). By the Markov property, the joint likelihood
of these observations can be written as a product of the transition densities:
$$
    p(y \mid \theta) = \prod_{i=0}^{n-1} p(\Delta_i, y_{i+1} \mid y_i, \theta),
$$
where $\Delta_i=t_{i+1}-t_i$ is the time between observations, $y_i$ is the $i$-th observed
state, $y_{i+1}$ is the $i+1$ observed state $(i=0,\ldots,n-1)$, and $\theta$ contains the diffusion parameters.

The likelihood $p(\Delta_i, \cdot \mid \cdot, \theta)$ is usually intractable for 
this class of models. This motivates the idea of approximating the transition densities.

#### Approximating the Transition Densities

Since for many diffusions the transition density is intractable, we need a way 
When the interval between two observed states is sufficiently small, one can use
the Euler approximation to the transition density. The Euler approximation to the 
transition density is a manageable Gaussian transition.

In order to shrink said interval,
we divide each observation interval into $M$ substeps of length
$$
h_i=\frac{\Delta_i}{M}.
$$
For interval $i$, we write
$$
u_{i,0}=y_i,
\qquad
u_{i,M}=y_{i+1},
\qquad
u_i=(u_{i,1},\ldots,u_{i,M-1}).
$$
The endpoints are observed and fixed. The $M-1$ intermediate states form the
unobserved discretized path. Under this notation, the Euler approximation to the
transition density is:
$$
    p^{(M)}(\Delta_i, y_{i+1} \mid y_i, \theta) = 
    \int \prod_{m=0}^{M-1}\phi(u_{m+1};\, u_m + h_i(\mu(u_m, \theta), h\nu(u_m, \theta)))du_1,\ldots, du_{M-1},
$$
where, $\phi(\cdot;\, \mu, \nu)$ denotes the Gaussian density with mean $\mu$ and
covariance $\nu$. While the overall integral is still difficult to compute, the factors
inside the integral are tractable. We therefore can estimate the overall integral using 
importance sampling by generating intermediate paths and averaging their weights.

#### Estimating the Transition Density Via Importance Sampling

### The Algorithm

### Example: Ornstein–Uhlenbeck Process